This is a text Markdown Cell. This is where we write stuff that involve decriptions, definitions, and explanations.

Remember:
Each Markdown Cell can only contain, at Max, 100 words. And in total, all of the markdown cells that we have created, combined, cannot exceed 5000 words.

In [ ]:
This is a code cell. Notice how words such as and, or, if, else, and else if, glow?
This is because it matches Python's syntax. 

Remember:
For each code cell, we can only have: 30 Lines Of Code Max (Excluding Blank Spaces) per Cell.
And we cannot have any comments either as the markdown cells will be used for explaining the code instead.

## Step 1: Import Libraries

In [24]:
import pandas as pd 
import numpy as np 
from sklearn.model_selection import train_test_split
SEED = 55

Import libraries for loading and cleaning data as well as splitting the dataset

## Data Cleaning, Preprocessing and EDA

### Cleaning

In [25]:
raw = pd.read_csv("dataset_Cheat_Detection.csv", sep=";", decimal=",")

Read the file and define that ; is a separator and , is the decimal point because the dataset has European format

In [26]:
cleaned = raw.drop(columns=['timestamp', 'isAimbot_max', 'isVidaInfinita_max', 'isMunicionInfinita_max', 'isSuperVelocidad_max'])

Remove the **Timestamp** Column because nearly every row has a unique value, so it works more as an identifier rather than a feature. The model may end up learning a time-based pattern instead of looking at the other feature values to decide whether cheating occurred or not.

**isAimbot_max, isVidaInfinita_max, isMunicionInfinita_max, and isSuperVelocidad_max** were also removed because they are sub-labels of **es_trampa_max**, which is 1 (Cheating) whenever at least one of them is true. Keeping them would leak the target to the model, and they would not be available in a real-world scenario.

In [27]:
cleaned.isna().sum()

angle_error_mean          0
angle_error_max           0
speed_mean                0
speed_max                 0
is_shot_sum               0
is_hit_sum                0
is_received_damage_sum    0
health_mean               0
ammo_left_min             0
ammo_max                  0
es_trampa_max             0
accuracy_rate             0
infinite_ammo_ratio       0
dtype: int64

Check if there are any cells with missing values that may affect the model's training or lead to errors during preprocessing.

In [28]:
dup_count = cleaned.duplicated().sum()
print ("Duplicates: ", dup_count)
if dup_count > 0:   
    cleaned.drop_duplicates(subset=None, inplace=True)

print(f'Before: {len(raw)} rows')
print(f'Target: {len(cleaned)} rows')

Duplicates:  60
Before: 3584 rows
Target: 3524 rows


Check if there are any duplicated rows that could lead to the model developing a bias and also preventing a leakage. There were 60 duplicates found since we removed the Timestamp column.


In [37]:
features = cleaned.drop(columns = "es_trampa_max")

print("Min:", features.min().min())
print("Max:", features.max().max())
print("All features in [0,1]:", ((features >= 0) & (features <= 1)).all().all(), "\n")

print("Label values:", sorted(cleaned["es_trampa_max"].unique()), "\n")

Min: 0.0
Max: 1.0
All features in [0,1]: True 

Label values: [np.int64(0), np.int64(1)] 



Checked for misrepresented values. All features fall within [0, 1], showing the data was already scaled by the source, and the label only contains 0 and 1. No invalid values were found.

### Pre-Processing

In [30]:
cleaned["es_trampa_max"] = cleaned["es_trampa_max"].astype(int)
print(cleaned["es_trampa_max"].dtype)

int64


The label es_trampa_max was stored as a decimal (0.0 or 1.0). It was changed to a whole number (0 or 1) because it only marks two groups, Not Cheating (0) and Cheating (1), rather than measuring an amount.

In [31]:
cleaned.to_csv("cleanedDataset.csv", index=False)

Save changes into the cleaned csv file. Below is the list of Pre-Processing techniques that were not applied:

Normalization: all features are already scaled to [0, 1]

One-hot encoding: Features are numeric, with no categorical columns.

Feature engineering: accuracy_rate (hits ÷ shots) and infinite_ammo_ratio (share of records where ammo didn't decrease) were provided by the source, so no new features were created.

Outliers: Extreme values come from Cheaters, removing them would delete the patterns the model needs to learn.

Standardization: Not applied at this stage. It will be fitted on the training set only, after splitting, to prevent data leakage.

### Exploratory Data Analysis

In [ ]:
rawAnalysis = raw.describe()
cleanedAnalysis = cleaned.describe()

compare = pd.concat([rawAnalysis, cleanedAnalysis], axis = 1, keys=['Raw Dataset', 'Cleaned Dataset'])
compare = compare.swaplevel(0, 1, axis = 1).sort_index(axis = 1)

label = 'es_trampa_max'
raw_counts = raw[label].value_counts()
cleaned_counts = cleaned[label].value_counts()

raw_pct = raw[label].value_counts(normalize=True) * 100
cleaned_pct = cleaned[label].value_counts(normalize=True) * 100

label_compare = pd.concat([raw_counts, raw_pct, cleaned_counts, cleaned_pct], axis=1, keys=['Raw Count', 'Raw %', 'Cleaned Count', 'Cleaned %'])

rawClassMeans = raw[cleaned.columns].groupby("es_trampa_max").mean(numeric_only=True).T
cleanedClassMeans = cleaned.groupby("es_trampa_max").mean().T

classCompare = pd.concat([rawClassMeans, cleanedClassMeans], axis=1,
                         keys=["Raw Dataset", "Cleaned Dataset"])
print(compare, "\n\n")
print(label_compare, "\n\n")
print(classCompare)

#### Comparison of Raw and Cleaned Datasets

A summary analysis using describe() was implemented for both the Raw and Cleaned datasets to compare changes. The cleaned dataset has fewer rows, from 3584 to 3524, because 60 duplicates were removed after dropping the timestamp and the four other sub-labels. 

The table confirms that the statistics are nearly unchanged. Every feature's mean shifted less than 0.01, and the standard deviations change by less than 0.004. The medians stayed relatively the same. This shows that removing the duplicate rows and irrelevant columns did not distort the distribution of the features and the balance between Cheating and Not Cheating

The label **es_trampa_max** values were also counted and showed that the ratio of Cheating and Not Cheating classes were preserved. Cheating went from 48.35% in the Raw Dataset to 48.52% in the Cleaned Dataset. Accuracy would also be a reliable evaluation metric.

#### Observations and Challenges

1. Balanced classes: The split is very close to 50/50, so the model would not require adding or subtracting samples or instances.
2. Skewed Features: Observing the dataset, many features have a lot of zeros, resulting in a positive skew distribution. For example, **angle_error_mean** has a mean of 0.093, but its median is at 0.007. Most normal players stay near zero, and the higher values mostly come from cheaters.
3. Gap between Classes: The mean for Cheaters is much higher on multiple features compared to Non-Cheaters. The largest differences are:\
       a. **ammo_left_min** (0.517 vs 0.213)\
       b. **infinite_ammo_ratio** (0.275 vs 0.038)\
       c. **angle_error_max** (0.252 vs 0.015)\
    These three feature may be the biggest indicators for Cheating, which suggests the model can perform well

## Dataset Split

In [33]:
x = cleaned.drop(columns = "es_trampa_max")
y = cleaned["es_trampa_max"]

x_train, x_temp, y_train, y_temp = train_test_split(x, y, test_size = 0.30, random_state = SEED, stratify = y)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size = 0.50, random_state = SEED, stratify = y_temp)

x contains all features while y contains **es_trampa_max**. The data was split twice to get 70/15/15. The first split keeps 70% of training and holds the 30% in a temporary set. The second split cuts the temporary set in half to give both the validation set and training set a 15% split. Both split functions use stratify so every set keeps about 48.5% of Cheaters. Both also use **random_state = SEED** so the split comes out the same every time.

In [34]:
splits = {"Train": y_train, "Validation": y_val, "Test": y_test}

summary = pd.DataFrame({
    name: {
        "Rows": len(s),
        "Not Cheating": (s == 0).sum(),
        "Cheating": (s == 1).sum(),
        "% Cheating": round(s.mean() * 100, 2)
    }
    for name, s in splits.items()
}).T

print(summary)

              Rows  Not Cheating  Cheating  % Cheating
Train       2466.0        1269.0    1197.0       48.54
Validation   529.0         273.0     256.0       48.39
Test         529.0         272.0     257.0       48.58


The Cheating share stays between 48.39% and 48.58% in all three sets, close to the 48.52% in the cleaned dataset. This confirms the stratified split kept the classes balanced.

In [36]:
pd.concat([x_train, y_train], axis=1).to_csv("trainSet.csv", index=False)
pd.concat([x_val, y_val], axis=1).to_csv("validationSet.csv", index=False)
pd.concat([x_test, y_test], axis=1).to_csv("testSet.csv", index=False)

Each set is saved as its own CSV so both models are trained and tested on exactly the same rows.

#### Data Leakage Prevention

1. The timestamp and the four sub-labels were removed before splitting, so no column gives away the answer.
2. Duplicates were removed before splitting, so the same row cannot be in both the training and test sets.
3. Any scaler will be fitted on the training set only, then applied to validation and test.
4. The validation set is used to tune hyperparameters. The test set is used only once, for the final evaluation.

#### Limitation

Each row summarizes only 2 seconds of one player's game, so one play session is split into many back-to-back rows. Rows from the same moment look almost the same because the player's speed, aim, and health would barely change in seconds. A random split can place one of these rows in training and the next in testing, which is a form of data leakage. The model is then tested on rows it has nearly seen before, so the test score may be higher than its true performance on new players and could hide overfitting.